## Libraries and constants

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

START = '2015-01-01'
END = '2024-12-31'

YEARLY_TRADING_DATES = 252 # n
MONTHLY_TRADING_DATES = YEARLY_TRADING_DATES / 12 # = 21 trading days/month
T_FACTOR = MONTHLY_TRADING_DATES / YEARLY_TRADING_DATES # (1+r/n)^{t_factor*n}

## Yahoo Finance scraping and Feature engineering

In [75]:
# Inspiration from https://www.kaggle.com/code/alessandrozanette/s-p500-analysis-using-yfinance-data
sp500 = yf.Ticker("^GSPC").history(period="12y") # need extra year for rolling returns in volatility calculations 
sp500 = sp500.drop(columns=['Dividends', 'Stock Splits']) # NULL columns
sp500 = sp500.drop(columns=['Open', 'High', 'Low']) # Only need close price to calc returns, keep volume

# sp500.tail() # shows the last values
# sp500.describe() # count, max, min, std etc

# Inspiration: Source - https://stackoverflow.com/a/50563791
# Posted by jezrael, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-24, License - CC BY-SA 4.0
sp500['daily_return'] = sp500['Close'].div(sp500['Close'].shift(1))-1
# use df.loc for indexing https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html
sp500.loc['2016-04-25 00:00:00-04:00','daily_return'] = 0

# Inspiration from https://www.kaggle.com/code/alessandrozanette/s-p500-analysis-using-yfinance-data
# Volatility
sp500['volatility'] = sp500['daily_return'].rolling(window=252).std() * (252 ** 0.5)

# Source - https://stackoverflow.com/a/72240529
# Posted by jinx, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-24, License - CC BY-SA 4.0
sp500.reset_index(inplace=True)
sp500['Date'] = sp500['Date'].dt.strftime('%Y-%m-%d')
sp500['Date'] = pd.to_datetime(sp500['Date'])

sp500 = sp500[sp500['Date'] > '2016-04-23'] # remove older dates
sp500.reset_index(inplace=True)
sp500 = sp500.drop(columns=['index'])

# Inspiration: Source - https://stackoverflow.com/a/57620144
# Posted by Chris
# Retrieved 2026-04-24, License - CC BY-SA 4.0
sp500['compounded_return (%)'] = ((sp500['daily_return'].add(1)).cumprod()-1)*100
sp500


,Date,Close,Volume,daily_return,volatility,compounded_return (%)
0,2016-04-25,2087.790039,3319740000,0.000000,0.166437,0.000000
1,2016-04-26,2091.699951,3557190000,0.001873,0.166395,0.187275
2,2016-04-27,2095.149902,4100110000,0.001649,0.166380,0.352519
3,2016-04-28,2075.810059,4309840000,-0.009231,0.166596,-0.573812
4,2016-04-29,2065.300049,4704720000,-0.005063,0.166363,-1.077215
...,...,...,...,...,...,...
2510,2026-04-20,7109.140137,4661130000,-0.002374,0.130631,240.510301
2511,2026-04-21,7064.009766,5160270000,-0.006348,0.130850,238.348667
2512,2026-04-22,7137.899902,4919460000,0.010460,0.128795,241.887822
2513,2026-04-23,7108.399902,5307260000,-0.004133,0.126673,240.474845
